In [1]:
 
# CELL 1: Imports and Paths
 
import os
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Set paths (matching 01_dataset_analysis.ipynb)
DATA_DIR = Path('./data/raw')
PATIENT_DIR = Path("./data/raw/AngioCAD_Dataset")
FEATURES_FILE = DATA_DIR / 'AngioCAD_Features.xlsx'
LABELS_FILE = DATA_DIR / 'AngioCAD_Labels.xlsx'
OUTPUT_DIR = Path('./data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: data\processed


In [2]:
  
# CELL 2: Load Raw Data
  

df_features = pd.read_excel(FEATURES_FILE)
df_labels = pd.read_excel(LABELS_FILE)

print(f"Features shape: {df_features.shape}")
print(f"Labels shape  : {df_labels.shape}")
df_features.head()

Features shape: (377, 60)
Labels shape  : (413, 18)


,ID,Sex,Age,Length,Weight,BMI,dominance,Right Coronary Series,Left Coronary Series,Diagnosis,...,2nd dig,Prox LCX,Mid LCX,Dist LCX,OM,Prox RCA,Mid RCA,Dist RCA,PDA,PLB
0,1,Male,52.0,170.0,82.0,28.373702,Right,1-2,3-9,STEMI,...,NL,NL,NL,NL,NL,1-25,NL,NL,NL,NL
1,2,Female,67.0,158.0,64.0,25.636917,Right,1-2,3,STEMI,...,NL,NL,NL,NL,NL,NL,26-50,NL,NL,76-90
2,3,Male,48.0,NaN,NaN,NaN,Right,2-3,4-5,STEMI,...,1-25,NL,NL,NL,NL,1-25,NL,NL,NL,NL
3,4,Male,47.0,176.0,70.0,22.598140,Right,7,1-6,STEMI,...,26-50,NL,NL,NL,1-25,26-50,91-99,NL,NL,91-99
4,5,Male,72.0,160.0,70.0,27.343750,Right,5-6,1-4,STEMI,...,NL,NL,1-25,NL,NL,1-25,76-90,1-25,NL,1-25


In [3]:
   
# CELL 3: Parse Stenosis Columns
   

STENOSIS_COLS = ['LM', 'Prox LAD', 'Mid LAD', 'Dist LAD', '1st dig', '2nd dig',
                 'Prox LCX', 'Mid LCX', 'Dist LCX', 'OM', 'Prox RCA',
                 'Mid RCA', 'Dist RCA', 'PDA', 'PLB']

def parse_stenosis(val):
    """Convert stenosis string values to numeric."""
    if pd.isna(val):
        return np.nan
    if val == 'NL':
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        if '-' in val:
            parts = val.split('-')
            try:
                return (float(parts[0]) + float(parts[1])) / 2
            except:
                return np.nan
        if val.strip().isdigit():
            return float(val)
    return np.nan

for col in STENOSIS_COLS:
    if col in df_labels.columns:
        df_labels[f'{col}_num'] = df_labels[col].apply(parse_stenosis)

NUM_STENOSIS_COLS = [f'{c}_num' for c in STENOSIS_COLS if f'{c}_num' in df_labels.columns]
print(f"Parsed {len(NUM_STENOSIS_COLS)} stenosis columns")

df_labels[NUM_STENOSIS_COLS].describe().round(2)

Parsed 15 stenosis columns


,LM_num,Prox LAD_num,Mid LAD_num,Dist LAD_num,1st dig_num,2nd dig_num,Prox LCX_num,Mid LCX_num,Dist LCX_num,OM_num,Prox RCA_num,Mid RCA_num,Dist RCA_num,PDA_num,PLB_num
count,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00,413.00
mean,2.02,23.08,48.34,6.02,23.41,7.64,12.47,11.83,4.77,24.05,22.20,24.43,10.54,6.80,5.63
std,9.11,35.26,41.93,21.22,36.38,23.45,26.22,27.64,19.55,36.29,32.74,35.67,25.88,21.67,20.49
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.00,0.00,63.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,0.00,38.00,95.00,0.00,38.00,0.00,13.00,0.00,0.00,38.00,38.00,38.00,0.00,0.00,0.00
max,83.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


In [4]:
    
# CELL 4: Create Target Labels
    

# Max stenosis per patient
df_labels['max_stenosis'] = df_labels[NUM_STENOSIS_COLS].max(axis=1)

# Binary CAD label (>= 50%)
df_labels['has_cad'] = (df_labels['max_stenosis'] >= 50).astype(int)

# Severity classification
def get_severity(x):
    if x == 0:      return 'Normal'
    elif x < 30:    return 'Mild'
    elif x < 50:    return 'Mild-Moderate'
    elif x < 70:    return 'Moderate'
    else:           return 'Severe'

df_labels['severity'] = df_labels['max_stenosis'].apply(get_severity)

# Number of diseased vessels
df_labels['num_diseased_vessels'] = df_labels[NUM_STENOSIS_COLS].ge(50).sum(axis=1)

print("\nBinary CAD distribution:")
print(df_labels['has_cad'].value_counts())
print(f"\nCAD prevalence: {df_labels['has_cad'].mean()*100:.2f}%")


Binary CAD distribution:
has_cad
1    324
0     89
Name: count, dtype: int64

CAD prevalence: 78.45%


In [5]:
     
# CELL 5: Merge Features + Labels
     

df = pd.merge(df_features, df_labels, on='ID', how='inner')
print(f"Merged shape: {df.shape}")

Merged shape: (377, 96)


In [6]:
      
# CELL 6: Define Feature Columns
      

CLINICAL_NUMERIC = [
    'Age', 'Length', 'Weight', 'BMI',
    'LVEF', 'LVEF dysfunction',
    'TG', 'Cholesterol', 'HDL', 'LDL', 'FBS',
    'Cr', 'BUN', 'Hgb', 'Hct',
    'leukocytes', 'Neut', 'Monocyte', 'MXD', 'Lymph',
    'PLT', 'RDW', 'Gensini Score',
]

CLINICAL_CATEGORICAL = [
    'Sex', 'dominance', 'Diagnosis',
    'DM', 'HTN', 'Hyperlipidemia', 'Current Smoker',
    'PVD', 'IHD', 'Stroke',
    'Beta blocker', 'Alpha blocker', 'Diuretic',
    'ACEI', 'ARB', 'CCB', 'ASA', 'Clopidogrel', 'Astatine',
]

# Keep only existing columns
CLINICAL_NUMERIC = [c for c in CLINICAL_NUMERIC if c in df.columns]
CLINICAL_CATEGORICAL = [c for c in CLINICAL_CATEGORICAL if c in df.columns]

print(f"Numeric features    : {len(CLINICAL_NUMERIC)}")
print(f"Categorical features: {len(CLINICAL_CATEGORICAL)}")

Numeric features    : 22
Categorical features: 19


In [8]:
       
# CELL 7: Handle Missing Values
       

# Numeric - median imputation
num_imputer = SimpleImputer(strategy='median')
df[CLINICAL_NUMERIC] = num_imputer.fit_transform(df[CLINICAL_NUMERIC])

# Categorical - mode imputation
for col in CLINICAL_CATEGORICAL:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values remaining: {df.isna().sum().sum()}")

Missing values remaining: 4


In [10]:
         
# CELL 8: Encode Categorical Variables
         

label_encoders = {}

for col in CLINICAL_CATEGORICAL:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

ENC_CATEGORICAL = [f'{c}_enc' for c in CLINICAL_CATEGORICAL]

# Show encoding mappings
for col in CLINICAL_CATEGORICAL[:3]:
    print(f"{col}: {list(label_encoders[col].classes_)}")

Sex: ['Female', 'Male']
dominance: ['Left', 'Right']
Diagnosis: ['NSTEMI', 'SA', 'STEMI', 'UA']


In [11]:
          
# CELL 9: Build Feature Matrix and Scale
          

FEATURE_COLS = CLINICAL_NUMERIC + ENC_CATEGORICAL

X = df[FEATURE_COLS].copy()
y_binary = df['has_cad'].values
y_severity = df['severity'].values

print(f"Feature matrix shape: {X.shape}")

# Scale numeric
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[CLINICAL_NUMERIC] = scaler.fit_transform(X[CLINICAL_NUMERIC])

print(f"Scaled shape: {X_scaled.shape}")

Feature matrix shape: (377, 41)
Scaled shape: (377, 41)


In [12]:
           
# CELL 10: Save Processed Data & Artifacts
           

df.to_csv(OUTPUT_DIR / 'clinical_features_merged.csv', index=False)
X_scaled.to_csv(OUTPUT_DIR / 'X_clinical_scaled.csv', index=False)
np.save(OUTPUT_DIR / 'y_binary.npy', y_binary)
np.save(OUTPUT_DIR / 'y_severity.npy', y_severity)

# Save artifacts for inference
with open(OUTPUT_DIR / 'preprocessing_artifacts.pkl', 'wb') as f:
    pickle.dump({
        'num_imputer'      : num_imputer,
        'scaler'           : scaler,
        'label_encoders'   : label_encoders,
        'feature_cols'     : FEATURE_COLS,
        'numeric_cols'     : CLINICAL_NUMERIC,
        'categorical_cols' : CLINICAL_CATEGORICAL,
        'stenosis_cols'    : STENOSIS_COLS,
    }, f)

print("✅ Preprocessing complete!")
print(f"   Saved to: {OUTPUT_DIR}")
print(f"   Files: clinical_features_merged.csv, X_clinical_scaled.csv")
print(f"          y_binary.npy, y_severity.npy, preprocessing_artifacts.pkl")

✅ Preprocessing complete!
   Saved to: data\processed
   Files: clinical_features_merged.csv, X_clinical_scaled.csv
          y_binary.npy, y_severity.npy, preprocessing_artifacts.pkl
